# ConvNeXt-Tiny — Face-Crop Inference + Grad-CAM Demo

# ConvNeXt-Tiny — Face-Crop Inference Demo

Mọi input inference đều đi qua **face detection + face crop** trước khi vào ConvNeXt-Tiny.

- Ảnh: `image → detect face → crop → 224×224 → normalize → model`
- Video: `video → sample frame → detect face → crop → model → temporal aggregation`
- Nếu không detect được mặt, frame/ảnh đó **không được đưa full-frame vào model**.
- Nếu có nhiều mặt, mặc định chọn mặt lớn nhất.
- **Không sử dụng TEST** trong notebook này. Threshold phải được lấy từ VAL.

In [ ]:
!pip install -q opencv-python-headless matplotlib pandas tqdm

import os, cv2, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.models import convnext_tiny

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# YuNet face detector
YUNET_PATH = "/content/face_detection_yunet_2023mar.onnx"
YUNET_URL = "https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx"

if not os.path.exists(YUNET_PATH):
    urllib.request.urlretrieve(YUNET_URL, YUNET_PATH)

face_detector = cv2.FaceDetectorYN.create(
    YUNET_PATH, "", (320, 320),
    score_threshold=0.65,
    nms_threshold=0.3,
    top_k=5000
)
print("✓ YuNet ready")

In [ ]:
FACE_SCORE_THRESHOLD = 0.65
FACE_MARGIN = 0.20
MIN_FACE_SIZE = 40

def detect_faces_bgr(image_bgr):
    h, w = image_bgr.shape[:2]
    face_detector.setInputSize((w, h))
    _, detections = face_detector.detect(image_bgr)

    if detections is None:
        return []

    faces = []
    for det in detections:
        x, y, bw, bh = map(int, det[:4])
        score = float(det[-1])
        if score < FACE_SCORE_THRESHOLD:
            continue
        x1, y1 = max(0, x), max(0, y)
        x2, y2 = min(w, x+bw), min(h, y+bh)
        if x2-x1 >= MIN_FACE_SIZE and y2-y1 >= MIN_FACE_SIZE:
            faces.append({
                "box": (x1,y1,x2,y2),
                "score": score,
                "area": (x2-x1)*(y2-y1)
            })
    return faces

def crop_largest_face(image_bgr):
    faces = detect_faces_bgr(image_bgr)
    if not faces:
        return None, None, []

    selected = max(faces, key=lambda z: z["area"])
    x1,y1,x2,y2 = selected["box"]
    h,w = image_bgr.shape[:2]

    mx = int(FACE_MARGIN*(x2-x1))
    my = int(FACE_MARGIN*(y2-y1))

    cx1, cy1 = max(0,x1-mx), max(0,y1-my)
    cx2, cy2 = min(w,x2+mx), min(h,y2+my)

    crop = image_bgr[cy1:cy2, cx1:cx2].copy()
    info = {
        "detector_box": (x1,y1,x2,y2),
        "crop_box": (cx1,cy1,cx2,cy2),
        "score": selected["score"],
        "num_faces": len(faces)
    }
    return crop, info, faces

def draw_crop_box(image_bgr, info):
    vis = image_bgr.copy()
    if info:
        x1,y1,x2,y2 = info["crop_box"]
        cv2.rectangle(vis,(x1,y1),(x2,y2),(0,255,0),2)
        cv2.putText(vis, f"face score={info['score']:.2f}",
                    (x1,max(25,y1-8)), cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,(0,255,0),2)
    return vis

In [ ]:
# Load EXP-6 ConvNeXt-Tiny checkpoint
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive"
CHECKPOINT_PATH = os.path.join(
    DRIVE_ROOT,
    "FaceForensics_Experiments",
    "finetune_convnext_tiny",
    "checkpoints",
    "convnext_tiny_exp6_aug_gentle_fullft_best.pth"
)

IMG_SIZE = 224
BATCH_SIZE = 32
THRESHOLD = 0.50   # SAU EXP-6: thay bằng threshold tối ưu trên VAL

class ConvNeXtHead(nn.Module):
    def __init__(self, in_features, dropout=0.25, use_batchnorm=False):
        super().__init__()
        self.bn = nn.BatchNorm1d(in_features) if use_batchnorm else nn.Identity()
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(in_features, 2, bias=True)

    def forward(self, x):
        return self.fc(self.dropout(self.bn(x)))

model = convnext_tiny(weights=None)
in_features = model.classifier[-1].in_features
model.classifier = nn.Sequential(
    model.classifier[0],
    model.classifier[1],
    ConvNeXtHead(in_features, dropout=0.25, use_batchnorm=False)
)

ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
state = ckpt.get("model_state_dict", ckpt)
model.load_state_dict(state, strict=True)
model = model.to(device).eval()

print("✓ Checkpoint loaded")
print("Epoch:", ckpt.get("epoch", "unknown"))

In [ ]:
# Exact evaluation preprocessing after face crop
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

def predict_face_crops(face_crops_bgr):
    tensors = []
    for crop in face_crops_bgr:
        rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        tensors.append(eval_transform(Image.fromarray(rgb)))

    probs = []
    for start in range(0, len(tensors), BATCH_SIZE):
        x = torch.stack(tensors[start:start+BATCH_SIZE]).to(device)
        if device.type == "cuda":
            x = x.contiguous(memory_format=torch.channels_last)
        with torch.no_grad():
            p_fake = torch.softmax(model(x), dim=1)[:,1]
        probs.extend(p_fake.cpu().numpy().tolist())
    return np.asarray(probs)

## Grad-CAM — mô hình đang nhìn vào đâu?

Grad-CAM được thêm **chỉ ở inference**, không tham gia fine-tuning và không làm thay đổi weights của best model.

Pipeline ảnh:

`Ảnh → Face Detection → Face Crop → ConvNeXt-Tiny → P(FAKE) → Grad-CAM`

Grad-CAM target mặc định là **class FAKE (index = 1)**, giúp kiểm tra những vùng trên **face crop thực sự đưa vào model** đang đóng góp mạnh vào quyết định FAKE.

In [ ]:
# ============================================================
# GRAD-CAM FOR CONVNEXT-TINY
# ============================================================

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None

        self.forward_handle = target_layer.register_forward_hook(
            self._forward_hook
        )
        self.backward_handle = target_layer.register_full_backward_hook(
            self._backward_hook
        )

    def _forward_hook(self, module, inputs, output):
        self.activations = output.detach()

    def _backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def remove(self):
        self.forward_handle.remove()
        self.backward_handle.remove()

    def __call__(self, x, target_class=1):
        self.model.zero_grad(set_to_none=True)

        # Grad-CAM cần gradient; không dùng torch.no_grad() ở đây.
        logits = self.model(x)
        target_score = logits[:, target_class].sum()
        target_score.backward()

        activations = self.activations
        gradients = self.gradients

        # Global-average gradient theo spatial dimensions
        weights = gradients.mean(dim=(2, 3), keepdim=True)

        # Weighted sum của feature maps
        cam = torch.relu((weights * activations).sum(dim=1))

        # Normalize CAM từng ảnh về [0, 1]
        cam_min = cam.flatten(1).min(dim=1)[0].view(-1, 1, 1)
        cam_max = cam.flatten(1).max(dim=1)[0].view(-1, 1, 1)
        cam = (cam - cam_min) / (cam_max - cam_min + 1e-8)

        probabilities = torch.softmax(logits, dim=1)

        return (
            cam.cpu().numpy(),
            probabilities.detach().cpu().numpy()
        )


# ConvNeXt-Tiny final spatial feature map.
# Không target Linear classifier vì Grad-CAM cần thông tin không gian.
TARGET_LAYER = model.features[-1]
gradcam = GradCAM(model, TARGET_LAYER)

print("✓ Grad-CAM ready")
print("Target layer:", TARGET_LAYER)

In [ ]:
# ============================================================
# GRAD-CAM HELPERS
# ============================================================

def prepare_face_tensor(face_crop_bgr):
    rgb = cv2.cvtColor(face_crop_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    return eval_transform(pil).unsqueeze(0).to(device)


def run_gradcam_on_face(face_crop_bgr, target_class=1):
    x = prepare_face_tensor(face_crop_bgr)

    model.eval()

    # Không dùng torch.no_grad() vì Grad-CAM cần gradient.
    cam, probs = gradcam(x, target_class=target_class)

    return (
        cam[0],
        float(probs[0, 0]),
        float(probs[0, 1])
    )


def make_gradcam_overlay(face_crop_bgr, cam, alpha=0.45):
    h, w = face_crop_bgr.shape[:2]

    cam_uint8 = np.uint8(np.clip(cam, 0, 1) * 255)

    heatmap = cv2.applyColorMap(
        cam_uint8,
        cv2.COLORMAP_JET
    )

    heatmap = cv2.resize(
        heatmap,
        (w, h),
        interpolation=cv2.INTER_LINEAR
    )

    overlay = cv2.addWeighted(
        face_crop_bgr,
        1.0 - alpha,
        heatmap,
        alpha,
        0
    )

    return heatmap, overlay

## Demo 1 — Ảnh + Face Crop + Grad-CAM

Đây là cell quan trọng nhất để debug trường hợp **ảnh webcam REAL nhưng model dự đoán FAKE**.

Hãy kiểm tra heatmap có tập trung vào mắt, mũi, miệng, da hay lại tập trung vào tóc, viền crop/background.

In [ ]:
# ============================================================
# IMAGE INFERENCE + FACE CROP + GRAD-CAM
# ============================================================

IMAGE_PATH = "/content/your_image.jpg"

image_bgr = cv2.imread(IMAGE_PATH)

if image_bgr is None:
    raise FileNotFoundError(IMAGE_PATH)

# 1. Detect + crop face
face_crop, info, faces = crop_largest_face(image_bgr)

if face_crop is None:
    raise RuntimeError(
        "Không detect được khuôn mặt. "
        "Thử ảnh rõ mặt hơn hoặc giảm FACE_SCORE_THRESHOLD."
    )

# 2. Prediction
cam, p_real, p_fake = run_gradcam_on_face(
    face_crop,
    target_class=1       # FAKE
)

label = "FAKE" if p_fake >= THRESHOLD else "REAL"

# 3. Visualize
heatmap, overlay = make_gradcam_overlay(
    face_crop,
    cam,
    alpha=0.45
)

print("=" * 70)
print("IMAGE + FACE CROP + GRAD-CAM")
print("=" * 70)
print("Detected faces :", len(faces))
print(f"Face score     : {info['score']:.4f}")
print(f"P(REAL)        : {p_real:.4f}")
print(f"P(FAKE)        : {p_fake:.4f}")
print(f"Threshold      : {THRESHOLD:.4f}")
print(f"Prediction     : {label}")
print("=" * 70)

fig, ax = plt.subplots(1, 4, figsize=(18, 5))

ax[0].imshow(
    cv2.cvtColor(
        draw_crop_box(image_bgr, info),
        cv2.COLOR_BGR2RGB
    )
)
ax[0].set_title("Original + detected face")
ax[0].axis("off")

ax[1].imshow(
    cv2.cvtColor(
        face_crop,
        cv2.COLOR_BGR2RGB
    )
)
ax[1].set_title("Face crop → model input")
ax[1].axis("off")

ax[2].imshow(
    cv2.cvtColor(
        heatmap,
        cv2.COLOR_BGR2RGB
    )
)
ax[2].set_title("Grad-CAM for FAKE")
ax[2].axis("off")

ax[3].imshow(
    cv2.cvtColor(
        overlay,
        cv2.COLOR_BGR2RGB
    )
)
ax[3].set_title(
    f"{label}\n"
    f"P(FAKE)={p_fake:.3f}"
)
ax[3].axis("off")

plt.tight_layout()
plt.show()

## Demo 2 — Video + Face Crop + Grad-CAM

Video vẫn thực hiện **face detection/crop trên từng sampled frame**.

Sau khi classifier dự đoán, notebook chọn các frame có `P(FAKE)` cao nhất rồi chạy Grad-CAM trên chính face crop của các frame đó.

In [ ]:
# ============================================================
# VIDEO: TOP FRAMES + FACE CROP + GRAD-CAM
# ============================================================

def collect_video_faces(
    video_path,
    frame_step=10,
    max_frames=150
):
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise FileNotFoundError(video_path)

    items = []
    frame_idx = 0
    sampled = 0

    while True:
        ok, frame = cap.read()

        if not ok or sampled >= max_frames:
            break

        if frame_idx % frame_step == 0:
            sampled += 1

            crop, info, _ = crop_largest_face(frame)

            # Không fallback sang full frame.
            if crop is not None:
                items.append({
                    "frame_index": frame_idx,
                    "original_bgr": frame.copy(),
                    "face_crop_bgr": crop,
                    "detector_score": info["score"]
                })

        frame_idx += 1

    cap.release()
    return items


def show_top_video_gradcams(
    video_path,
    frame_step=10,
    max_frames=150,
    top_k=6
):
    items = collect_video_faces(
        video_path,
        frame_step=frame_step,
        max_frames=max_frames
    )

    if not items:
        raise RuntimeError(
            "Không detect được face trong video."
        )

    # Prediction cho tất cả face crops
    probs = predict_face_crops(
        [item["face_crop_bgr"] for item in items]
    )

    for item, p_fake in zip(items, probs):
        item["p_fake"] = float(p_fake)
        item["p_real"] = 1.0 - float(p_fake)

    # Chọn các frame model nghi ngờ FAKE nhất
    top_items = sorted(
        items,
        key=lambda x: x["p_fake"],
        reverse=True
    )[:top_k]

    n = len(top_items)

    fig, ax = plt.subplots(
        2,
        n,
        figsize=(4 * n, 8)
    )

    if n == 1:
        ax = np.array(ax).reshape(2, 1)

    for j, item in enumerate(top_items):

        cam, _, _ = run_gradcam_on_face(
            item["face_crop_bgr"],
            target_class=1
        )

        _, overlay = make_gradcam_overlay(
            item["face_crop_bgr"],
            cam,
            alpha=0.45
        )

        ax[0, j].imshow(
            cv2.cvtColor(
                item["face_crop_bgr"],
                cv2.COLOR_BGR2RGB
            )
        )

        ax[0, j].set_title(
            f"Frame {item['frame_index']}\n"
            f"P(FAKE)={item['p_fake']:.3f}"
        )
        ax[0, j].axis("off")

        ax[1, j].imshow(
            cv2.cvtColor(
                overlay,
                cv2.COLOR_BGR2RGB
            )
        )

        ax[1, j].set_title(
            "Grad-CAM for FAKE"
        )
        ax[1, j].axis("off")

    plt.tight_layout()
    plt.show()

    return pd.DataFrame([
        {
            "frame_index": item["frame_index"],
            "p_fake": item["p_fake"],
            "p_real": item["p_real"],
            "detector_score": item["detector_score"]
        }
        for item in top_items
    ])


# Example:
#
# VIDEO_PATH = "/content/your_demo_video.mp4"
#
# top_frames = show_top_video_gradcams(
#     VIDEO_PATH,
#     frame_step=10,
#     max_frames=150,
#     top_k=6
# )

## Demo 1 — Ảnh

**Ảnh nguyên không bao giờ được đưa trực tiếp vào model.**

`Upload image → YuNet → largest face → margin crop → ConvNeXt-Tiny`

In [ ]:
IMAGE_PATH = "/content/your_image.jpg"

image_bgr = cv2.imread(IMAGE_PATH)
if image_bgr is None:
    raise FileNotFoundError(IMAGE_PATH)

face_crop, info, faces = crop_largest_face(image_bgr)

if face_crop is None:
    raise RuntimeError("Không detect được khuôn mặt. Thử ảnh rõ mặt hơn hoặc giảm FACE_SCORE_THRESHOLD.")

p_fake = float(predict_face_crops([face_crop])[0])
p_real = 1.0 - p_fake
label = "FAKE" if p_fake >= THRESHOLD else "REAL"

print("Detected faces:", len(faces))
print(f"Detector score: {info['score']:.4f}")
print(f"P(REAL): {p_real:.4f}")
print(f"P(FAKE): {p_fake:.4f}")
print(f"Threshold: {THRESHOLD:.4f}")
print("Prediction:", label)

fig, ax = plt.subplots(1,2,figsize=(12,5))
ax[0].imshow(cv2.cvtColor(draw_crop_box(image_bgr,info),cv2.COLOR_BGR2RGB))
ax[0].set_title("Original + face crop")
ax[0].axis("off")
ax[1].imshow(cv2.cvtColor(face_crop,cv2.COLOR_BGR2RGB))
ax[1].set_title(f"Actual model input — {label}")
ax[1].axis("off")
plt.tight_layout()
plt.show()

## Demo 2 — Video

Mỗi sampled frame đều phải detect/crop face. Frame không có face sẽ bị bỏ qua, **không fallback sang full frame**.

In [ ]:
VIDEO_PATH = "/content/your_demo_video.mp4"

def infer_video(video_path, frame_step=10, max_frames=300, threshold=0.50):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(video_path)

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    face_crops, frame_ids, scores = [], [], []
    idx, sampled = 0, 0

    while True:
        ok, frame = cap.read()
        if not ok or sampled >= max_frames:
            break

        if idx % frame_step == 0:
            sampled += 1
            crop, info, _ = crop_largest_face(frame)

            if crop is not None:
                face_crops.append(crop)
                frame_ids.append(idx)
                scores.append(info["score"])

        idx += 1

    cap.release()

    if not face_crops:
        raise RuntimeError("Không detect được face trong video.")

    probs = predict_face_crops(face_crops)

    result = pd.DataFrame({
        "frame_index": frame_ids,
        "detector_score": scores,
        "fake_probability": probs
    })
    result["prediction"] = (result["fake_probability"] >= threshold).astype(int)

    mean_fake = float(probs.mean())
    median_fake = float(np.median(probs))
    fake_ratio = float(result["prediction"].mean())
    label = "FAKE" if mean_fake >= threshold else "REAL"

    print("="*60)
    print("VIDEO INFERENCE")
    print("Total frames:", total)
    print("Sampled:", sampled)
    print("Face detected:", len(result))
    print(f"Detection rate: {len(result)/max(sampled,1):.2%}")
    print(f"Mean P(FAKE): {mean_fake:.4f}")
    print(f"Median P(FAKE): {median_fake:.4f}")
    print(f"Fake-frame ratio: {fake_ratio:.2%}")
    print(f"Threshold: {threshold:.4f}")
    print("FINAL:", label)
    print("="*60)

    return result, label

results, final_label = infer_video(
    VIDEO_PATH,
    frame_step=10,
    max_frames=300,
    threshold=THRESHOLD
)

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(results["frame_index"], results["fake_probability"],
         marker="o", markersize=3)
plt.axhline(THRESHOLD, linestyle="--",
            label=f"Threshold={THRESHOLD:.2f}")
plt.xlabel("Video frame")
plt.ylabel("P(FAKE)")
plt.title(f"Frame-level prediction — FINAL: {final_label}")
plt.legend()
plt.grid(True)
plt.show()

## Protocol cuối

- Training: frame-level.
- Inference ảnh: **face detection → face crop → model**.
- Inference video: **frame → face detection → face crop → model → temporal aggregation**.
- Không đưa full image/frame vào ConvNeXt.
- Không dùng TEST để chọn threshold.
- Nếu webcam REAL vẫn cho `P(FAKE)` rất cao dù face crop đúng, đó là dấu hiệu cần kiểm tra **domain shift / dataset bias / calibration**, không nên chữa bằng cách bỏ face crop.